# 13. Quadratic Discriminant Analysis (QDA)
### Classification with Quadratic Boundaries

**Quadratic Discriminant Analysis (QDA)** extends LDA by relaxing the shared covariance assumption. Each class has its own individual covariance matrix $\Sigma_k$.

#### Key Concepts:
1. **Class-Specific Covariance**:
   $$X | Y=k \sim \mathcal{N}(\mu_k, \Sigma_k)$$
2. **Quadratic Decision Boundaries**:
   Because each class possesses a distinct covariance matrix, quadratic terms in the log-posterior do not cancel out, yielding quadratic curves (parabolas, hyperbolas, ellipses) as decision boundaries:
   $$\delta_k(x) = -\frac{1}{2} \ln|\Sigma_k| - \frac{1}{2}(x - \mu_k)^T \Sigma_k^{-1}(x - \mu_k) + \ln P(Y = k)$$


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
# Step 2: Generate Non-linear Dataset where Covariances Differ
X, y = make_classification(
    n_samples=500, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.05, class_sep=1.2, random_state=42
)

# Induce different class covariances
X[y == 1] = X[y == 1] * [2.0, 0.5]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [ ]:
# Step 3: Train and Compare LDA vs QDA
lda = LinearDiscriminantAnalysis().fit(X_train, y_train)
qda = QuadraticDiscriminantAnalysis().fit(X_train, y_train)

print(f"LDA Test Accuracy: {accuracy_score(y_test, lda.predict(X_test)):.4f}")
print(f"QDA Test Accuracy: {accuracy_score(y_test, qda.predict(X_test)):.4f}")
print("
QDA Classification Report:
", classification_report(y_test, qda.predict(X_test)))


In [ ]:
# Step 4: Visualizing Decision Boundaries (Linear vs Quadratic)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))

# Plot LDA
Z_lda = lda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax1.contourf(xx, yy, Z_lda, alpha=0.3, cmap=plt.cm.coolwarm)
ax1.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k')
ax1.set_title(f"LDA (Linear Boundary) - Acc: {lda.score(X_test, y_test):.2f}")

# Plot QDA
Z_qda = qda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z_qda, alpha=0.3, cmap=plt.cm.coolwarm)
ax2.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k')
ax2.set_title(f"QDA (Quadratic Boundary) - Acc: {qda.score(X_test, y_test):.2f}")

plt.show()


### Summary
- **Strengths**: More flexible than LDA, captures different variances across classes without assuming shared covariance.
- **Weaknesses**: Estimates $K \times p(p+1)/2$ covariance parameters, requiring significantly more training samples to avoid singular matrix errors.
